## Introduction!
This dataset is the Kaggle Housing Dataset that's quite popular. It's being used for a kaggle tournament. The goal is to get a solid score. We'll clean the data, perform an EDA, prepare for modeling, and finally perform an experament to see which model did best and to use it in my Kaggle Tournament! The problem is that we have all these features of a home and now we need to predict the price based on those features. Here's how we'll do it.
- Data Cleaning
- EDA
- Modeling
- Experiments
    - Experiment 1
    - Experiment 2
    - Experiment 3
- Deciding which model is better.
- Sources

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

SVM

In [ ]:
df_train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
df_test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
df_test.head()

In [ ]:
df_train.columns, df_train.shape

### Types of Regressions Used

We’ll focus on two powerful ensemble regression methods: **Random Forest Regression** and **XGBoost**.

---

####  Random Forest Regression
Random Forest Regression is an **ensemble learning method** that builds multiple decision trees and combines their results to improve accuracy and reduce overfitting.  
Each tree in the forest is trained on a **random subset of the data** and a **random subset of features**. The final prediction is obtained by averaging the predictions from all the trees.  

**Key Features:**
- Reduces variance compared to a single decision tree.  
- Handles nonlinear relationships and complex feature interactions.  
- Works well even when there are missing values or noisy data.  

**Formula (conceptually):**
\[
\hat{y} = \frac{1}{N} \sum_{i=1}^{N} T_i(x)
\]
where \( T_i(x) \) is the prediction of the *i-th* tree, and \( N \) is the total number of trees.

---

#### XGBoost (Extreme Gradient Boosting)
XGBoost is a **gradient boosting algorithm** that builds trees sequentially — each new tree corrects the errors made by the previous ones. It’s known for its **speed, regularization, and high predictive accuracy**.

**How it works:**
1. The model starts with an initial prediction (often the mean of the target).  
2. It calculates **residual errors** (differences between predicted and actual values).  
3. New trees are trained to predict these residuals.  
4. Each new tree’s predictions are combined with the previous ones using a **learning rate** to control the contribution.  
5. The process continues until the model reaches a stopping criterion (like a maximum number of trees or minimal improvement).  

**Key Features:**
- Incorporates **regularization** (L1 & L2) to prevent overfitting.  
- Efficient handling of missing data.  
- Highly optimized for performance and scalability.  

**Conceptual Formula:**
\[
\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta \cdot f_t(x_i)
\]
where \( f_t(x_i) \) is the new tree’s prediction at step *t*, and \( \eta \) is the learning rate.


There are too many columns ):

## Data Cleaning

In [ ]:
df_train.isna().sum()[df_train.isna().sum() !=0]

### I was going through the data description one by one, formulating how I could Fill NA's. It summarizes to this:
#### If it's a numeric, It's a zero, if it's categorical, it doesn't exist.

In [ ]:
df_train.info()

In [ ]:
Id = df_test.Id
df_test = df_test.drop('Id', axis = 1)
df_train = df_train.drop('Id', axis = 1)
train_categorical = df_train.select_dtypes(include = 'object').columns
train_numeric = df_train.select_dtypes(include=['int64', 'float64']).columns

df_train[train_categorical] = df_train[train_categorical].fillna("Not Applicable")
df_train[train_numeric] = df_train[train_numeric].fillna(0)

train_categorical = df_test.select_dtypes(include = 'object').columns
train_numeric = df_test.select_dtypes(include=['int64', 'float64']).columns

df_test[train_categorical] = df_test[train_categorical].fillna("Not Applicable")
df_test[train_numeric] = df_test[train_numeric].fillna(0)


In [ ]:
Id

## Exploratory Data Analysis

In [ ]:
df_train['SalePrice'].describe()

In [ ]:
sns.histplot(df_train['SalePrice']);

In [ ]:
var = 'GrLivArea'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)
data.plot.scatter(x=var, y='SalePrice', ylim=(0,800000));

In [ ]:
var = 'TotalBsmtSF'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)
data.plot.scatter(x=var, y='SalePrice', ylim=(0,800000));

In [ ]:
var = 'OverallQual'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)
f, ax = plt.subplots(figsize=(8, 6))
fig = sns.boxplot(x=var, y="SalePrice", data=data)
fig.axis(ymin=0, ymax=800000);

In [ ]:
var = 'YearBuilt'
data = pd.concat([df_train['SalePrice'], df_train[var]], axis=1)
f, ax = plt.subplots(figsize=(16, 8))
fig = sns.boxplot(x=var, y="SalePrice", data=data)
fig.axis(ymin=0, ymax=800000);
plt.xticks(rotation=90);

### So many dang columns

In [ ]:
sns.set()
cols = ['SalePrice', 'OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'FullBath', 'YearBuilt']
sns.pairplot(df_train[cols], size = 2.5)
plt.show();

idea: use a random forest regression model to find the feature importance, then prune for SVM

## Modeling

### Preparing the Dataset for Machine Learning
To get our dataset ready for modeling, we’ll perform the following steps:
1. **Separate the target variable** from the rest of the dataset.  
2. **Standardize all numeric columns** to ensure consistent scaling.  
3. **Create dummy variables** for all categorical features using one-hot encoding.  
4. **Split the dataset** into training and testing sets to evaluate model performance.


In [ ]:
# The training data
from sklearn.preprocessing import StandardScaler 
from sklearn.model_selection import train_test_split

y = df_train['SalePrice']
X = pd.get_dummies(df_train.drop('SalePrice', axis=1))

scaler = StandardScaler()
X[train_numeric] = scaler.fit_transform(X[train_numeric])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

In [ ]:
# test dataset
scaler = StandardScaler()
df_test[train_numeric] = scaler.fit_transform(df_test[train_numeric])
df_test = pd.get_dummies(df_test) 

## Experiments
- Idea 1 is to use the Random Forest Regression and seeing how that does
- Idea 2 is to use the top 80% of importance, then running GXBoosting
- Idea 3 is to use PCA to reduce the number of columns, then running GXBoosting the same as idea 2 to see if it improves accuracy.

### Experiment 1 

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np

rf = RandomForestRegressor(random_state=2)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
rf_r2 = r2_score(y_test, rf_pred)
mae = mean_absolute_error(y_test, rf_pred)
rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print(f"R² Score: {rf_r2:.4f}")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")

# Plot Actual vs Predicted
plt.scatter(y_test, rf_pred)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("Actual vs Predicted SalePrice (Random Forest)")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

Results:
- **$R^2 = .8834$** means that there's about a 89% accuracy
- **Mean Absolute Error (MAE)** of **\$16,482.65** means that, on average, the model’s predictions are off by about **\$16,482**, which is solid performance when home prices are in the **hundreds of thousands**.
- **Root Mean Squared Error** of **\$26,739.12** mean's that the price is almost garenteed to be accurate withing \$26,739.12.

### Experiment 2

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending = False)
importance_df['ImportancePercent'] = (
    importance_df['Importance'] / importance_df['Importance'].sum() * 100
)

importance_df['CumulativePercent'] = importance_df['ImportancePercent'].cumsum()
features_to_keep = importance_df[importance_df['CumulativePercent'] <= 80]['Feature']
features_to_keep

We will now build a model using those columns

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

X_important_Features = X_train[features_to_keep]

param_grid = {
    'n_estimators': [200, 400, 800],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0],
    'reg_lambda': [1, 5, 10],   # L2
    'reg_alpha': [0, 0.1, 1.0], # L1
}

xgb = XGBRegressor()
grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    verbose=1,
    n_jobs=-1,
    refit=True
)

grid.fit(X_important_Features, y_train)

gxb_pred = grid.predict(X_test[features_to_keep])

gxb_r2 = r2_score(y_test, gxb_pred)
mae = mean_absolute_error(y_test, gxb_pred)
rmse = np.sqrt(mean_squared_error(y_test, gxb_pred))
print("Best CV R²:", grid.best_score_)
print("Best params:", grid.best_params_)
 
print(f"R² Score: {gxb_r2:.4f}")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")

plt.scatter(y_test, gxb_pred)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("Actual vs Predicted SalePrice (GXBoost)")
plt.legend()
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

### Experiment 3
 

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=.95)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

X_train_pca = pd.DataFrame(
    X_train_pca,
    columns=[f"PC{i+1}" for i in range(X_train_pca.shape[1])]
)
X_test_pca = pd.DataFrame(
    X_test_pca,
    columns=[f"PC{i+1}" for i in range(X_test_pca.shape[1])]
)

In [ ]:
X_train_pca

In [ ]:
rf = RandomForestRegressor(random_state=2)
rf.fit(X_train, y_train)
rf_pca_pred = rf.predict(X_test)


pca_r2 = r2_score(y_test, rf_pca_pred)
mae = mean_absolute_error(y_test, rf_pca_pred)
rmse = np.sqrt(mean_squared_error(y_test, rf_pca_pred))

print(f"R² Score: {pca_r2:.4f}")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")

plt.scatter(y_test, rf_pca_pred)
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("Actual vs Predicted SalePrice (GXBoost)")
plt.legend()
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()


The random forest XGBoost performed the best.

In [ ]:
best_param = grid.best_params_

X_important_Features = X_train[features_to_keep]

xgb = XGBRegressor(**best_param)

xgb.fit(X_important_Features, y_train)
gxb_pred = xgb.predict(df_test[features_to_keep])

In [ ]:
output = pd.DataFrame({"Id" : Id, "SalePrice" : gxb_pred})

output.to_csv("submission.csv", index = False)
print("Done!")

## Impact

While using **XGBoost** or similar machine learning models to predict home prices can improve accuracy and efficiency for individual agents or investors, the **collective impact** of such models on the housing market raises serious concerns.  

As more businesses adopt predictive models for pricing homes, these models begin to **interact with each other** in a form of **algorithmic feedback loop**.  
When one company’s model uses market prices — which themselves may have been influenced by other models — as part of its training data, the predictions begin to **recursively reflect not just underlying economic fundamentals, but also the outputs of other algorithms**.  

Over time, this creates **model-on-model inflation**, where estimated prices are anchored not to intrinsic value, but to other **algorithmic expectations**.


## Conclusion

The **Random Forest Regression XGBoost** approach proved to be highly effective in improving predictive performance and model pruning.  
Among all the experiments conducted, this hybrid method achieved the **highest R² score**, indicating its superior ability to capture complex, nonlinear relationships in housing data.  

By combining the ensemble strengths of Random Forests with the boosting precision of XGBoost, the model was able to **minimize overfitting**, enhance generalization, and deliver **robust, interpretable predictions**.  

However, while these results are promising at the individual level, the broader adoption of similar predictive models across the industry should be approached cautiously. As discussed earlier, widespread use of algorithmic pricing models could contribute to **feedback-driven market distortions**, emphasizing the importance of **ethical deployment and regulatory oversight** in future applications.

## Sources
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- https://www.geeksforgeeks.org/machine-learning/xgboost-for-regression/
- https://machinelearningmastery.com/regression-metrics-for-machine-learning/
- https://xgboosting.com/
- https://dl.acm.org/doi/abs/10.1145/3447755

